# Episode 10: RoPE: rotary position embeddings from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep010-rope.ipynb)

We replace our model's table of positions with rotary position embeddings, which turn each query and key by its position, so attention scores depend on how far apart two tokens are, and the KV cache can slide past the context window.

[The video](https://youtu.be/CVeEPW67Gv8) · [The episode's page](https://mvgpt.nenuadrian.com/ep010-rope/) · [The docs](https://nenuadrian.github.io/mvGPT/rope/) · [The code at `ep010/end`](https://github.com/nenuadrian/mvGPT/tree/ep010/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep010/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("pretrain")

## The data

In [ ]:
!python data.py  # TinyStories, the 10,000-word vocabulary, and the stories as token ids

## Pretraining with rotary position embeddings
Ten minutes, to `checkpoints/pretrain-rope`.

In [ ]:
!python pretrain.py 10 --rope

## Against episode two's model

In [ ]:
!python evaluate.py checkpoints/pretrain/model.pt checkpoints/pretrain-rope/model.pt --out results/rope

## Past the window
The loss by position, past the 256 tokens both trained on: on the CPU, then on the GPU.

In [ ]:
!python context.py checkpoints/pretrain/model.pt checkpoints/pretrain-rope/model.pt --device cpu
!python context.py checkpoints/pretrain/model.pt checkpoints/pretrain-rope/model.pt

Next: [episode 11, BPE in C++: a byte-level tokenizer for a GPT](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep011-bpe.ipynb)